In [23]:
import pandas as pd
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
def convert(text):
    try:
        return [i['name'] for i in ast.literal_eval(text)]
    except:
        return []

def preprocess():
    movies = pd.read_csv(r"C:\Users\Lonovo\OneDrive\Documents\Movie Recommendation System\movie_dataset.csv")

    # Handle missing values
    movies.dropna(subset=['genres', 'director'], inplace=True)

    # Convert genres
    movies['genres'] = movies['genres'].apply(convert)

    # Convert director to list
    movies['director'] = movies['director'].apply(lambda x: [str(x)])

    # Combine tags
    movies['tags'] = movies['genres'] + movies['director']

    # Clean tags
    movies['tags'] = movies['tags'].apply(lambda x: [str(i).replace(" ", "") for i in x])
    movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))

    # Final dataframe
    df = movies[['id', 'title', 'tags']].copy()
    df = df.rename(columns={'id': 'movie_id'})

    return df

In [25]:
df = preprocess()
print(df.head())

   movie_id                                     title              tags
0     19995                                    Avatar      JamesCameron
1       285  Pirates of the Caribbean: At World's End     GoreVerbinski
2    206647                                   Spectre         SamMendes
3     49026                     The Dark Knight Rises  ChristopherNolan
4     49529                               John Carter     AndrewStanton


In [26]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
vectors = tfidf.fit_transform(df['tags']).toarray()

In [27]:
similarity = cosine_similarity(vectors)

In [28]:
def recommend(movie):
    if movie not in df['title'].values:
        print(" Movie not found!")
        return

    movie_index = df[df['title'] == movie].index[0]
    distances = similarity[movie_index]

    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

    print(f"\n🎬 Recommendations for {movie}:\n")
    for i in movies_list:
        print(df.iloc[i[0]].title)

In [29]:
recommend("Avatar")


🎬 Recommendations for Avatar:

Titanic
Terminator 2: Judgment Day
True Lies
The Abyss
Aliens
